In [1]:
import os
os.environ["CALITP_BQ_MAX_BYTES"] = str(800_000_000_000) ## 800GB?

from calitp.tables import tbl
from calitp import query_sql
import calitp.magics
import branca

import shared_utils
import utils

from siuba import *
import pandas as pd
import geopandas as gpd
import shapely

import datetime as dt
import time
from zoneinfo import ZoneInfo

import rt_analysis as rt
import importlib

import gcsfs
fs = gcsfs.GCSFileSystem()

from tqdm import tqdm_notebook
from tqdm.notebook import trange, tqdm

/opt/conda/lib/python3.9/site-packages/geopandas/_compat.py:111: UserWarning: The Shapely GEOS version (3.10.2-CAPI-1.16.0) is incompatible with the GEOS version PyGEOS was compiled with (3.10.1-CAPI-1.16.0). Conversions between both will be slow.
  warnings.warn(


In [2]:
analysis_date = dt.date(2022, 5, 18)

In [3]:
bbb_vp = utils.get_vehicle_positions(300, analysis_date)

In [4]:
bbb_vp

,calitp_itp_id,calitp_url_number,header_timestamp,vehicle_timestamp,entity_id,vehicle_id,trip_id,vehicle_longitude,vehicle_latitude
1,300,0,2022-05-18 00:01:59,2022-05-18 00:01:20,None,5319,881449,-118.489070,34.014805
2,300,0,2022-05-18 00:02:59,2022-05-18 00:02:05,None,5319,881449,-118.489070,34.014805
3,300,0,2022-05-18 00:03:59,2022-05-18 00:03:33,None,5319,881449,-118.489070,34.014805
4,300,0,2022-05-18 00:06:00,2022-05-18 00:05:02,None,5319,881449,-118.489070,34.014805
5,300,0,2022-05-18 00:08:01,2022-05-18 00:07:15,None,5319,881449,-118.489070,34.014805
...,...,...,...,...,...,...,...,...,...
144990,300,0,2022-05-18 16:54:39,2022-05-18 16:54:12,None,5316,883352,-118.449980,33.992640
144991,300,0,2022-05-18 16:55:39,2022-05-18 16:54:56,None,5316,883352,-118.449970,33.992626
144992,300,0,2022-05-18 16:56:39,2022-05-18 16:56:25,None,5316,883352,-118.448265,33.990800
144993,300,0,2022-05-18 16:57:39,2022-05-18 16:57:10,None,5316,883352,-118.447685,33.990154


In [5]:
bbb_vp.header_timestamp.min()

Timestamp('2022-05-18 00:00:58')

In [6]:
bbb_vp.header_timestamp.max()

Timestamp('2022-05-19 00:30:11')

In [7]:
rt.OperatorDayAnalysis?

Init signature: rt.OperatorDayAnalysis(itp_id, analysis_date, pbar=None)
Docstring:     
New top-level class for rt delay/speed analysis of a single operator on a single day
    
File:           ~/data-analyses/rt_delay/rt_analysis.py
Type:           type
Subclasses:     


In [8]:
pbar = tqdm()

0it [00:00, ?it/s]

In [9]:
bbb = rt.OperatorDayAnalysis(300, analysis_date, pbar)

found parquet
getting trips...


/home/jovyan/data-analyses/_shared_utils/shared_utils/utils.py:38: UserWarning: this is an initial implementation of Parquet/Feather file support and associated metadata.  This is tracking version 0.1.0 of the metadata specification at https://github.com/geopandas/geo-arrow-spec

This metadata specification does not yet make stability promises.  We do not yet recommend using this in a production setting unless you are able to rewrite your Parquet/Feather files.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', message='.*initial implementation of Parquet.*')
/home/jovyan/data-analyses/_shared_utils/shared_utils/utils.py:38: UserWarning: this is an initial implementation of Parquet/Feather file support and associated metadata.  This is tracking version 0.1.0 of the metadata specification at https://github.com/geopandas/geo-arrow-spec

This metadata specification does not yet make stability promises.  We do not yet recommend using this in a 

less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 1km of data
less than 

In [18]:
bbb.rt_trips >> filter(_.route_short_name == 'R12') >> arrange(_.median_time)

,calitp_itp_id,calitp_url_number,service_date,trip_key,trip_id,route_id,direction_id,shape_id,calitp_extracted_at,calitp_deleted_at,route_short_name,route_long_name,route_desc,route_type,median_time,direction,mean_speed_mph,calitp_agency_name
693,300,0,2022-05-18,6797445014565992395,882488,3489,0,25914,2022-03-29,2099-01-01,R12,Venice/Westwood Sta/UCLA Rapid,None,3,05:44:13,Northbound,3.117748,Big Blue Bus
745,300,0,2022-05-18,730747023174550181,882487,3489,0,25914,2022-03-29,2099-01-01,R12,Venice/Westwood Sta/UCLA Rapid,None,3,05:46:21,Northbound,2.444321,Big Blue Bus
817,300,0,2022-05-18,4764541779728689532,882489,3489,0,25914,2022-03-29,2099-01-01,R12,Venice/Westwood Sta/UCLA Rapid,None,3,05:47:25,Northbound,1.808605,Big Blue Bus
753,300,0,2022-05-18,-8039951267135081885,882564,3489,1,25915,2022-03-29,2099-01-01,R12,Venice/Westwood Sta/UCLA Rapid,None,3,06:15:47.500000,Southbound,7.868440,Big Blue Bus
742,300,0,2022-05-18,-8918690479222221636,882565,3489,1,25915,2022-03-29,2099-01-01,R12,Venice/Westwood Sta/UCLA Rapid,None,3,06:25:19,Southbound,12.032122,Big Blue Bus
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
787,300,0,2022-05-18,2703106464280695975,882549,3489,0,25914,2022-03-29,2099-01-01,R12,Venice/Westwood Sta/UCLA Rapid,None,3,21:28:11.500000,Northbound,8.583710,Big Blue Bus
722,300,0,2022-05-18,2303633253298786684,882637,3489,1,25915,2022-03-29,2099-01-01,R12,Venice/Westwood Sta/UCLA Rapid,None,3,21:43:28,Southbound,9.896840,Big Blue Bus
761,300,0,2022-05-18,-7519516594928499227,882550,3489,0,25914,2022-03-29,2099-01-01,R12,Venice/Westwood Sta/UCLA Rapid,None,3,21:50:46,Northbound,8.880197,Big Blue Bus
686,300,0,2022-05-18,-1277444804726595602,882638,3489,1,25915,2022-03-29,2099-01-01,R12,Venice/Westwood Sta/UCLA Rapid,None,3,21:57:01,Southbound,13.044645,Big Blue Bus


In [19]:
bbb.stop_delay_view >> filter(_.route_short_name == 'R12') >> arrange(_.arrival_time)

,stop_id,stop_name,geometry,shape_id,shape_meters,trip_key,trip_id,stop_sequence,arrival_time,route_id,route_short_name,direction_id,actual_time,delay,delay_seconds
12,389,WESTWOOD NB & SANTA MONICA FS,POINT (144250.612 -439381.439),25914,4746.227266,730747023174550181,882487,10,2022-05-18 05:42:25,3489,R12,0,2022-05-18 05:45:27.255291,0 days 00:03:02.255291,182
2,398,WESTWOOD NB & WILSHIRE NS,POINT (143653.876 -438472.971),25914,5836.526072,730747023174550181,882487,11,2022-05-18 05:45:56,3489,R12,0,2022-05-18 05:47:58.388640,0 days 00:02:02.388640,122
9,402,WESTWOOD NB & WEYBURN NS,POINT (143514.338 -438054.558),25914,6302.085356,730747023174550181,882487,12,2022-05-18 05:47:26,3489,R12,0,2022-05-18 05:49:14.877352,0 days 00:01:48.877352,108
8,1149,WESTWOOD PLAZA NB & J STEIN EYE FS,POINT (143512.235 -437768.687),25914,6588.004521,730747023174550181,882487,13,2022-05-18 05:48:21,3489,R12,0,2022-05-18 05:50:46.002498,0 days 00:02:25.002498,145
12,389,WESTWOOD NB & SANTA MONICA FS,POINT (144250.612 -439381.439),25914,4746.227266,6797445014565992395,882488,10,2022-05-18 05:54:25,3489,R12,0,2022-05-18 05:52:25.456440,0 days 00:00:00,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6,1539,OVERLAND SB & PALMS FS,POINT (146702.019 -442219.319),25915,6373.150104,3623285453416580945,882639,12,2022-05-18 22:30:03,3489,R12,1,2022-05-18 22:35:12.884411,0 days 00:05:09.884411,309
3,1566,OVERLAND SB & CHARNOCK NS,POINT (146841.784 -442429.205),25915,6625.248133,3623285453416580945,882639,13,2022-05-18 22:30:39,3489,R12,1,2022-05-18 22:35:55.716318,0 days 00:05:16.716318,316
0,1541,VENICE EB & OVERLAND FS,POINT (147179.666 -442835.122),25915,7200.603999,3623285453416580945,882639,14,2022-05-18 22:32:02,3489,R12,1,2022-05-18 22:37:44.646700,0 days 00:05:42.646700,342
1,1542,MOTOR SB & WASHINGTON NS,POINT (147533.635 -442850.687),25915,7711.017542,3623285453416580945,882639,15,2022-05-18 22:33:16,3489,R12,1,2022-05-18 22:38:49.548586,0 days 00:05:33.548586,333


In [14]:
bbb.position_interpolators['882321']['rt'].detailed_speed_map()

/home/jovyan/data-analyses/rt_delay/rt_analysis.py:108: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.



In [13]:
bbb.position_interpolators['882321']['rt']

SyntaxError: invalid syntax (2209976205.py, line 1)